In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!gunzip /content/drive/MyDrive/ADR/meddra_all_se.tsv.gz

In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
import torch


tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")
model = AutoModel.from_pretrained("dmis-lab/biobert-base-cased-v1.1")


df = pd.read_csv("/content/drive/MyDrive/ADR/meddra_all_se.tsv", sep="\t", header=None,
                 names=["STITCH_CID", "UMLS_ID", "UMS_Concept_ID", "Source", "ADR_NAME"])

adr_list = df['UMLS_ID'].dropna().drop_duplicates().tolist()
print(df.shape)
def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[0][0].numpy()

adr_embeddings = []
valid_adr_names = []

for adr in adr_list:
    try:
        emb = get_bert_embedding(adr)
        adr_embeddings.append(emb)
        valid_adr_names.append(adr)
    except:
        continue


np.save("/content/drive/MyDrive/ADR/adr_bert_embeddings.npy", np.array(adr_embeddings))
# pd.DataFrame({'ADR': valid_adr_names}).to_csv("/content/drive/MyDrive/ADR/adr_names.csv", index=False)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


(309849, 5)


In [ ]:
# df = pd.read_csv("/content/drive/MyDrive/ADR/meddra_all_se.tsv", sep="\t", header=None)
# df.head()
# adr_list
pd.DataFrame({'ADR': valid_adr_names}).to_csv("/content/drive/MyDrive/ADR/adr_names.csv", index=False)

In [ ]:
# import pandas as pd


# df = pd.read_csv("/content/drive/MyDrive/ADR/meddra_all_se.tsv", sep="\t", header=None,
#                  names=["STITCH_CID", "UMLS_ID", "UMS_Concept_ID", "Source", "ADR_NAME"])


# adr_df = df[["ADR_NAME"]].drop_duplicates().reset_index(drop=True)
# adr_df.columns = ["ADR"]
# adr_df.to_csv("/content/drive/MyDrive/ADR/adr_names.csv", index=False)
# df.head()

In [ ]:
import pandas as pd


df = pd.read_csv("/content/drive/MyDrive/ADR/meddra_all_se.tsv", sep="\t", header=None,
                 names=["old_CID","STITCH_CID", "UMLS_ID", "UMS_Concept_ID", "Source", "ADR_NAME"])


print(df.head())
df["CID"] = df["STITCH_CID"].str.extract(r"CID(\d+)").astype(float).astype(int)
df["pub_chem_CID"] = df["old_CID"].str.extract(r"CID(\d+)").astype(float).astype(int)
cid_df = df[["old_CID","STITCH_CID", "CID","pub_chem_CID"]].drop_duplicates().reset_index(drop=True)


cid_df.to_csv("/content/drive/MyDrive/ADR/drug_cids.csv", index=False)
cid_df


        old_CID    STITCH_CID   UMLS_ID UMS_Concept_ID    Source  \
0  CID100000085  CID000010917  C0000729            LLT  C0000729   
1  CID100000085  CID000010917  C0000729             PT  C0000737   
2  CID100000085  CID000010917  C0000737            LLT  C0000737   
3  CID100000085  CID000010917  C0000737             PT  C0687713   
4  CID100000085  CID000010917  C0000737             PT  C0000737   

                ADR_NAME  
0       Abdominal cramps  
1         Abdominal pain  
2         Abdominal pain  
3  Gastrointestinal pain  
4         Abdominal pain  


,old_CID,STITCH_CID,CID,pub_chem_CID
0,CID100000085,CID000010917,10917,100000085
1,CID100000119,CID000000119,119,100000119
2,CID100000137,CID000000137,137,100000137
3,CID100000143,CID000000143,143,100000143
4,CID100000143,CID000006006,6006,100000143
...,...,...,...,...
1551,CID156603655,CID056603655,56603655,156603655
1552,CID156842239,CID056842239,56842239,156842239
1553,CID170683024,CID070683024,70683024,170683024
1554,CID170695640,CID070695640,70695640,170695640


In [ ]:
!gunzip /content/drive/MyDrive/ADR/CID-SMILES.gz

^C


In [ ]:
import pandas as pd


cid_df = pd.read_csv("/content/drive/MyDrive/ADR/drug_cids.csv")

our_cids = set(cid_df["pub_chem_CID"].astype(str).str.lstrip('0'))


In [ ]:

# link to CID-SMILES.gz file: https://drive.google.com/file/d/1DfgWLqc2AUupA5wPYV__O9umZG03KMDX/view?usp=drive_link
# OR: https://ftp.ncbi.nlm.nih.gov/pubchem/Compound/Extras/

chunks = pd.read_csv("/content/drive/MyDrive/ADR/CID-SMILES.gz", sep="\t", header=None, names=["CID", "SMILES"], compression='gzip', chunksize=100000)
drug_list = []
all_data = []
for chunk in chunks:

    chunk["CID"] = chunk["CID"].astype(str).str.lstrip('0')
    filtered = chunk[chunk["CID"].isin(our_cids)]
    all_data.append(filtered)


if all_data:
    drug_smiles_df = pd.concat(all_data)
else:
    drug_smiles_df = pd.DataFrame(columns=["CID", "SMILES"])

drug_smiles_df.to_csv("/content/drive/MyDrive/ADR/drug_smiles.csv", index=False)

print(f"Found {len(drug_smiles_df)} SMILES for given CIDs.")



Found 1357 SMILES for given CIDs.


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/ADR/CID-SMILES.gz", sep="\t", header=None, names=["CID", "SMILES"], compression="gzip", nrows=10)
print(df)


   CID                                     SMILES
0    1           CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C
1    2              CC(=O)OC(CC(=O)O)C[N+](C)(C)C
2    3                   C1=CC(C(C(=C1)C(=O)O)O)O
3    4                                    CC(CN)O
4    5                       C(C(=O)COP(=O)(O)O)N
5    6  C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl
6    7                     CCN1C=NC2=C(N=CN=C21)N
7    8                        CCC(C)(C(C(=O)O)O)O
8    9          C1(C(C(C(C(C1O)O)OP(=O)(O)O)O)O)O
9   11                                   C(CCl)Cl


In [ ]:
from transformers import RobertaTokenizer, RobertaModel
import torch
import numpy as np

#  SMILES strings in 'drug_smiles.csv' with column 'SMILES'
smiles_df = pd.read_csv("/content/drive/MyDrive/ADR/drug_smiles.csv")
tokenizer = RobertaTokenizer.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")
model = RobertaModel.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")

drug_embeddings = []
drug_ids = []

for cid, text in zip(smiles_df["CID"], smiles_df["SMILES"]):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        emb = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
        drug_embeddings.append(emb)
        drug_ids.append(cid)

# به آرایه numpy تبدیل کن
drug_embeddings = np.array(drug_embeddings)  # (n_drug, 768)
drug_ids = np.array(drug_ids)

# ذخیره در دو فایل جدا
# np.save("/content/drive/MyDrive/ADR/drug_ids.npy", drug_ids)
pd.DataFrame({"CID": drug_ids}).to_csv("/content/drive/MyDrive/ADR/drug_ids.csv", index=False)
np.save("/content/drive/MyDrive/ADR/drug_chemberta_embeddings.npy", drug_embeddings)

print("Saved:", drug_ids.shape, drug_embeddings.shape)


tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/179M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/179M [00:00<?, ?B/s]

Saved: (1357,) (1357, 768)
